In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# Load dataset
data = pd.read_csv("retail_customer_segmentation.csv")

# Remove missing values
clean_data = data.dropna()

# Feature columns
X = clean_data.drop(columns=["customer_id", "customer_segment"])

# Target variable
y = clean_data["customer_segment"]

# Convert categorical variables to numeric
X = pd.get_dummies(X, drop_first=True)

# Feature engineering
X["income_x_frequency"] = X["annual_income"] * X["purchase_frequency"]
X["income_x_order"] = X["annual_income"] * X["avg_order_value"]

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Polynomial features
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

# Hyperparameter tuning
param_grid = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train_poly, y_train)

# Best model
best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test_poly)

# Results
print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# F1 Score
f1 = f1_score(y_test, y_pred, average='weighted')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

print("\nConfusion Matrix:")
print(cm)

Best Params: {'metric': 'manhattan', 'n_neighbors': 9, 'weights': 'distance'}
Best CV Score: 0.6698768044646141
Accuracy: 0.6630052724077329
F1 Score: 0.6588793933572491

Confusion Matrix:
[[ 359  213   75   29]
 [  69  869  114  210]
 [  21   68 2401  558]
 [  20  180  744  898]]
